# Transformer 100M CUDA — Google Colab Training

**Sebelum mulai**, pastikan struktur Google Drive seperti ini:
```
MyDrive/
  transformer_100m_cuda.zip   ← source code
  data/
    corpus_final/
      english/...
      indonesian/...
  sft_data/
    sft_clean/...
    dpo/...
```

> **Urutan eksekusi:** Jalankan cell satu per satu dari atas ke bawah.


## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Config — Sesuaikan path di sini

In [ ]:
import os

# ╔══════════════════════════════════════════════════════════╗
# ║  SESUAIKAN PATH INI DENGAN LOKASI FILE DI GOOGLE DRIVE  ║
# ╚══════════════════════════════════════════════════════════╝
DRIVE_ROOT = "/content/drive/MyDrive"   # ganti jika ada subfolder, misal: "/content/drive/MyDrive/Projects"

ZIP_PATH   = f"{DRIVE_ROOT}/transformer_100m_cuda.zip"
DATA_PATH  = f"{DRIVE_ROOT}/data"        # folder berisi corpus_final/
SFT_PATH   = f"{DRIVE_ROOT}/sft_data"   # folder berisi sft_clean/, dpo/

# Working directory di Colab (lokal — lebih cepat dari Drive)
WORK_DIR   = "/content/transformer_100m_cuda"

# Checkpoint disimpan ke Drive agar persist saat Colab disconnect
CKPT_DRIVE = f"{DRIVE_ROOT}/checkpoints_100m"

# ── Export sebagai env vars agar bash cell bisa akses langsung ──
os.environ["WORK_DIR"]   = WORK_DIR
os.environ["ZIP_PATH"]   = ZIP_PATH
os.environ["DATA_PATH"]  = DATA_PATH
os.environ["SFT_PATH"]   = SFT_PATH
os.environ["CKPT_DRIVE"] = CKPT_DRIVE

# ── Validasi file ada di Drive ──
errors = []
if not os.path.exists(ZIP_PATH):
    errors.append(f"  ✗ ZIP tidak ditemukan : {ZIP_PATH}")
if not os.path.isdir(DATA_PATH):
    errors.append(f"  ✗ data/ tidak ditemukan: {DATA_PATH}")
if not os.path.isdir(SFT_PATH):
    errors.append(f"  ✗ sft_data/ tidak ditemukan: {SFT_PATH}")

if errors:
    print("❌ PATH SALAH — periksa kembali DRIVE_ROOT di atas:")
    for e in errors: print(e)
else:
    print("✓ Semua path ditemukan")
    print(f"  ZIP      : {ZIP_PATH}")
    print(f"  data/    : {DATA_PATH}")
    print(f"  sft_data/: {SFT_PATH}")
    print(f"  ckpt/    : {CKPT_DRIVE}")


## 3. Extract source code ke local disk (lebih cepat)

In [ ]:
import os, shutil

# Clean previous run jika ada
if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)
    print(f"Cleaned old {WORK_DIR}")

# Buat folder tujuan dulu, lalu unzip isinya langsung ke sana
os.makedirs(WORK_DIR, exist_ok=True)
!unzip -q "$ZIP_PATH" -d "$WORK_DIR"

# Verify
assert os.path.exists(f"{WORK_DIR}/CMakeLists.txt"), \
    f"Extraction failed! File yang ada di {WORK_DIR}:\n" + str(os.listdir(WORK_DIR)[:10])
print(f"✓ Source extracted to {WORK_DIR}")


## 4. Symlink data folders (avoid copying large files)

In [ ]:
import os, shutil

def make_symlink(target, link):
    if os.path.islink(link):
        os.unlink(link)
    elif os.path.isdir(link):
        shutil.rmtree(link)
    elif os.path.exists(link):
        os.remove(link)
    os.symlink(target, link)
    print(f"✓ {os.path.basename(link)}/ -> {target}")

# Symlink data dari Drive ke working dir (tidak perlu copy)
make_symlink(DATA_PATH,  f"{WORK_DIR}/data")
make_symlink(SFT_PATH,   f"{WORK_DIR}/sft_data")

# Checkpoint dir di Drive (persist across sessions)
os.makedirs(CKPT_DRIVE, exist_ok=True)
make_symlink(CKPT_DRIVE, f"{WORK_DIR}/checkpoints")


## 5. Check GPU & CUDA version

In [ ]:
!nvidia-smi
!nvcc --version

## 6. Build the project

In [ ]:
%%bash
set -e
WORK_DIR="/content/transformer_100m_cuda"
cd "$WORK_DIR"

# Install build tools if missing
if ! command -v g++ &> /dev/null; then
    echo "Installing build-essential..."
    apt-get update -qq && apt-get install -y -qq build-essential cmake > /dev/null 2>&1
fi

# Ensure CUDA is in PATH
export PATH="/usr/local/cuda/bin:$PATH"
export LD_LIBRARY_PATH="/usr/local/cuda/lib64:$LD_LIBRARY_PATH"

# ─── Detect GPU -> CUDA compute capability ──────────────────────────────────
# NOTE: the old code passed -DGPU_TARGET, which is NOT read anywhere in
# CMakeLists.txt (0 matches). The variable that actually selects the target
# architecture is CMAKE_CUDA_ARCHITECTURES (see CMakeLists.txt:10-12).
GPU_NAME=$(nvidia-smi --query-gpu=name --format=csv,noheader | head -1)
echo "Detected GPU: $GPU_NAME"

if   echo "$GPU_NAME" | grep -qi "T4";   then CUDA_ARCH=75
elif echo "$GPU_NAME" | grep -qi "V100"; then CUDA_ARCH=70
elif echo "$GPU_NAME" | grep -qi "A100"; then CUDA_ARCH=80
elif echo "$GPU_NAME" | grep -qi "L4";   then CUDA_ARCH=89
elif echo "$GPU_NAME" | grep -qi "H100"; then CUDA_ARCH=90
else
    CUDA_ARCH=75
    echo "[WARN] Unrecognised GPU — defaulting to sm_75 (T4)."
fi

echo "Building for: sm_${CUDA_ARCH}"
echo "g++:  $(which g++)"
echo "nvcc: $(which nvcc)"

# Clean stale build cache (from a different machine / older source)
rm -rf build CMakeCache.txt
mkdir -p build && cd build
cmake .. -DCMAKE_BUILD_TYPE=Release -DCMAKE_CUDA_ARCHITECTURES=$CUDA_ARCH
make -j$(nproc)

echo ""
echo "Build complete"
ls -lh train_pretrain_cuda train_sft_cuda 2>/dev/null || true

# ═══════════════════════════════════════════════════════════════════════════
#  KERNEL CORRECTNESS CHECK — must pass before any training
# ═══════════════════════════════════════════════════════════════════════════
# grad_check is an opt-in target (EXCLUDE_FROM_ALL) so it is not produced by
# the `make` above; it has to be requested explicitly.
#
# It guards two bugs that previously ran for ~37k steps undetected, because
# the *reported* loss stayed accurate the whole time (the cross-entropy
# kernels use their own correct reduction, so only RMSNorm was affected):
#
#   1. block_reduce_sum dropped half of every reduction -> RMSNorm summed
#      384 of 768 dims while still dividing by the full D.
#   2. RMSNorm backward applied gamma outside the mean-subtraction.
#
# This cell runs with `set -e`, so a non-zero exit here STOPS the notebook
# and you never reach training with broken kernels. That is intentional.
echo ""
echo "═══════════════════════════════════════════════════════"
echo " Running grad_check (kernel correctness)"
echo "═══════════════════════════════════════════════════════"
# Guard: if the synced checkout is stale it will NOT have the grad_check
# target/CMakeLists entry. Fail with a clear message instead of a cryptic
# cmake "No rule to make target" error. If your checkout is stale, re-sync
# the latest code (commit that added cuda/grad_check.cu) before training.
if grep -q "add_executable(grad_check" "$WORK_DIR/CMakeLists.txt" \
   && [ -f "$WORK_DIR/cuda/grad_check.cu" ]; then
    cmake --build . --target grad_check -j$(nproc)
    ./grad_check
else
    echo "[GRAD_CHECK] STALE CHECKOUT: grad_check target/source not found."
    echo "[GRAD_CHECK] Re-sync the latest code (needs cuda/grad_check.cu + CMakeLists entry)."
    echo "[GRAD_CHECK] ABORTING -- kernels are UNVERIFIED on this checkout."
    exit 1
fi

# ─── Build inference binary (used by eval-during-training) ──────────────────
echo ""
echo "Building infer binary..."
rm -rf "$WORK_DIR/build_infer"
mkdir -p "$WORK_DIR/build_infer" && cd "$WORK_DIR/build_infer"
cmake "$WORK_DIR/infer_cpp" -DCMAKE_BUILD_TYPE=Release
make -j$(nproc)
echo "Infer build complete"
ls -lh infer 2>/dev/null || true

echo ""
echo "=== ALL BUILDS OK + KERNELS VERIFIED ==="

## 7. Start Pre-training

> ### ⚠️ Checkpoint lama TIDAK kompatibel
>
> Perbaikan pada `cuda/cuda_kernels.cu` mengubah cara RMSNorm menormalisasi:
>
> - **Sebelum:** `block_reduce_sum` membuang separuh hasil reduksi, sehingga
>   `mean(x²)` hanya menjumlah **384 dari 768 dimensi** tapi tetap dibagi `D`.
> - **Sesudah:** menjumlah seluruh 768 dimensi (terverifikasi eksak).
>
> Bobot pada checkpoint lama sudah **beradaptasi** ke normalizer yang rusak itu
> (`final_rms` bergeser ke mean 2.68, norma residual mencapai 460–870). Menjalankan
> bobot tersebut dengan kernel yang sudah benar adalah kombinasi terburuk — loss
> akan melonjak dan kamu tidak bisa tahu apakah perbaikannya berhasil.
>
> **Karena itu `--resume` sudah dihapus dari cell di bawah, dan ada guard yang
> menolak jalan kalau checkpoint lama masih ada.**
>
> Kalau guard-nya menyala, backup lalu kosongkan folder checkpoint:
>
> ```python
> import shutil, os
> os.makedirs(f"{CKPT_DRIVE}_old_broken", exist_ok=True)
> for f in os.listdir(CKPT_DRIVE):
>     shutil.move(f"{CKPT_DRIVE}/{f}", f"{CKPT_DRIVE}_old_broken/{f}")
> ```
>
> Training harus dimulai **dari nol** (step 0).

In [ ]:
import subprocess, os, glob

# ═══ GUARD: refuse to start if an incompatible checkpoint is present ═══
# The RMSNorm reducer fix changes the normalizer, so weights trained before it
# cannot be resumed. See the markdown cell above for the full explanation.
_ckpt_dir = f"{WORK_DIR}/checkpoints"
env = os.environ.copy()
env.pop('CUDA_VISIBLE_DEVICES', None)

os.makedirs(_ckpt_dir, exist_ok=True)

print("Starting pretraining (single GPU, from scratch)...")
print("=" * 60)

proc = subprocess.Popen(
    ['stdbuf', '-oL', './build/train_pretrain_cuda',
     '--data_dir',       'data/corpus_final',
     '--vocab_dir',      'vocab_6',
     '--bpe',            'bpe_v6.bin',
     '--checkpoint_dir', 'checkpoints',
     '--batch_size',     '24',
     '--max_seq',        '256',
     '--max_tokens',     '2000000000',
     '--lr',             '1.4e-4',
     '--save_every',     '150',
     '--log_every',      '5',
     '--grad_clip',      '5.0',
     '--loss_scale',     '4096',
     '--upload_every',   '300',
     '--safety',         'light',
     '--eval_every_tokens', '200000000',
     '--eval_infer_bin', './build_infer/infer'],
    cwd=WORK_DIR,
    env=env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    # Read raw BYTES, decode manually below.
    #
    # Do NOT use encoding=/errors= here: that wraps the pipe in a
    # TextIOWrapper whose line buffering is unreliable under Colab's
    # captured stdout, so output arrives in large delayed chunks (or not
    # at all until the process exits). Binary mode + explicit decode also
    # guarantees that partial multi-byte UTF-8 sequences from the JA eval
    # output can never raise UnicodeDecodeError mid-training.
    bufsize=0,
)

try:
    for raw in iter(proc.stdout.readline, b''):
        # errors='replace' keeps a truncated multi-byte char from killing
        # the loop; the training process itself is unaffected either way.
        print(raw.decode('utf-8', errors='replace'), end='', flush=True)
except KeyboardInterrupt:
    proc.terminate()
    print("\nTraining dihentikan manual (Ctrl+C).")

proc.wait()
if proc.returncode == 0:
    print("\nTraining selesai.")
elif proc.returncode == 2:
    print("\nTraining stopped: CUDA OOM. Turunkan --batch_size lalu ulangi.")
else:
    print(f"\nTraining exit code: {proc.returncode}")

## 7b. (Optional) Resume training jika Colab disconnect

`--resume` **hanya aman** untuk melanjutkan checkpoint yang dibuat **setelah**
perbaikan kernel RMSNorm. Jangan pernah dipakai pada checkpoint dari run lama.

Cara memastikan checkpoint-nya berasal dari run baru: `train_state.txt` harus
menunjukkan `step` yang dimulai dari 0 pada sesi pertama setelah perbaikan
(bukan melanjutkan ~37.000).

Kalau aman, tambahkan `'--resume'` ke daftar argumen di cell 7, lalu jalankan
ulang. Guard di cell tersebut perlu di-skip secara manual — karena guard itu
memang dirancang untuk memblokir checkpoint lama, dan tidak bisa membedakan
checkpoint lama dari checkpoint baru.

Yang perlu diperhatikan saat resume (`train_cuda.cu:965-985`): kalau `--lr`
atau `--grad_clip` berbeda dari nilai yang tersimpan di `train_state.txt`,
binary otomatis mengaktifkan warmup ulang selama 100 step. Itu perilaku yang
diinginkan, tapi jangan kaget kalau LR terlihat rendah di awal.

## 8. (Optional) Multi-GPU — DDP dengan 2 GPU

Colab Pro+ kadang bisa dapat 2 GPU. Jika ya, rebuild dengan DDP:

In [ ]:
import subprocess, sys, os, glob

NUM_GPUS = int(subprocess.check_output("nvidia-smi -L | wc -l", shell=True).strip())
print(f"Available GPUs: {NUM_GPUS}")

if NUM_GPUS >= 2:
    # ── Guard yang sama seperti cell 7 ──
    _ckpt_dir = f"{WORK_DIR}/checkpoints"
    _stale = glob.glob(f"{_ckpt_dir}/ckpt_pretrain*.safetensors") \
           + glob.glob(f"{_ckpt_dir}/ckpt_optim*.safetensors")
    if _stale:
        for f in sorted(_stale):
            print(f"   {f}")
        raise SystemExit("Checkpoint lama terdeteksi — lihat penjelasan di cell 7.")

    print("Multi-GPU detected! Rebuilding with DDP...")
    os.system("apt-get install -y libnccl2 libnccl-dev 2>/dev/null")

    # NOTE: -DGPU_TARGET tidak dibaca CMake; yang benar CMAKE_CUDA_ARCHITECTURES.
    build_cmd = (
        f"cd {WORK_DIR}/build && "
        f"cmake .. -DCMAKE_BUILD_TYPE=Release -DCMAKE_CUDA_ARCHITECTURES=75 -DUSE_DDP=ON && "
        f"make -j$(nproc)"
    )
    ret = os.system(build_cmd)
    if ret != 0:
        raise RuntimeError("DDP build GAGAL — jangan lanjut training.")

    # Guard stale checkout pada path multi-GPU juga.
    if os.path.exists(f"{WORK_DIR}/cuda/grad_check.cu") and \
       "add_executable(grad_check" in open(f"{WORK_DIR}/CMakeLists.txt").read():
        ret = os.system(
            f"cd {WORK_DIR}/build && cmake --build . --target grad_check -j$(nproc) && ./grad_check"
        )
        if ret != 0:
            raise RuntimeError("grad_check GAGAL — jangan lanjut training.")
        print("DDP build complete + kernels verified")
    else:
        raise RuntimeError(
            "STALE CHECKOUT: grad_check target/source tidak ditemukan. "
            "Re-sync kode terbaru (butuh cuda/grad_check.cu + entry CMakeLists)."
        )

    # launch_ddp.sh sudah menambahkan `--ddp --num_gpus 2` sendiri,
    # jadi jangan ditulis ulang di sini.
    #
    # Perbaikan dari versi lama:
    #   --seq_len      -> --max_seq   (--seq_len tidak dikenali binary; flag
    #                                  tak dikenal ditelan diam-diam tanpa error)
    #   --batch_size   64 -> 24       (64 OOM di T4 untuk model 204M; ini
    #                                  per-GPU, jadi efektifnya 48)
    #   --grad_clip / --loss_scale / --bpe ditambahkan agar konsisten cell 7
    #   --resume       dihapus        (checkpoint lama tidak kompatibel)
    cmd = [
        "bash", f"{WORK_DIR}/scripts/launch_ddp.sh",
        f"{WORK_DIR}/build/train_pretrain_cuda",
        "--data_dir",       "data/corpus_final",
        "--vocab_dir",      "vocab_6",
        "--bpe",            "bpe_v6.bin",
        "--checkpoint_dir", "checkpoints",
        "--batch_size",     "24",
        "--max_seq",        "256",
        "--max_tokens",     "2000000000",
        "--lr",             "2.0e-4",
        "--save_every",     "150",
        "--log_every",      "5",
        "--grad_clip",      "5.0",
        "--loss_scale",     "4096",
        "--safety",         "light",
    ]

    print("=" * 60)
    proc = subprocess.Popen(
        cmd,
        cwd=WORK_DIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        # Binary mode + manual decode, same rationale as cell 7: text mode
        # (universal_newlines/encoding) buffers unreliably under Colab's
        # captured stdout, and a truncated multi-byte JA char could raise.
        bufsize=0,
        env={**os.environ, "NUM_GPUS": str(NUM_GPUS)},
    )
    try:
        for raw in iter(proc.stdout.readline, b''):
            print(raw.decode('utf-8', errors='replace'), end="", flush=True)
    except KeyboardInterrupt:
        proc.terminate()
        print("\nTraining dihentikan manual.")
    proc.wait()

else:
    print("Single GPU terdeteksi — jalankan cell 7 (tanpa DDP)")

## 9. Monitor training progress

In [ ]:
# ── Training state (step, loss, tokens) ──
import glob, os

state_path = f"{WORK_DIR}/checkpoints/train_state.txt"
if os.path.exists(state_path):
    print("train_state.txt:")
    for line in open(state_path):
        line = line.strip()
        if line:
            print(f"   {line}")
    print()
else:
    print("Belum ada train_state.txt (training belum menyimpan checkpoint)\n")

# ── Checkpoint files ──
ckpts = sorted(glob.glob(f"{WORK_DIR}/checkpoints/*.safetensors"))
if ckpts:
    for c in ckpts:
        print(f"   {os.path.basename(c):40s} {os.path.getsize(c)/1e6:8.1f} MB")
else:
    print("No checkpoints yet")

# ── GPU utilization ──
print()
!nvidia-smi --query-gpu=utilization.gpu,memory.used,memory.total --format=csv